# Part 4: Classification

**Course:** 2026 KMITL Data Analytics

This notebook teaches classification: K-Nearest Neighbours, decision trees,
logistic regression and Support Vector Machines. Every idea starts with a small
numerical example, then runs real code and a chart.


## Learning objectives

By the end of this notebook you can:

1. Explain discrete class labels, binary and multiclass classification.
2. Use distance and majority voting in K-Nearest Neighbours.
3. Choose K on a validation set and read an accuracy-versus-K chart.
4. Describe root, decision and leaf nodes, entropy and information gain.
5. Train and visualize a decision tree.
6. Explain the sigmoid function, probability, threshold and decision boundary.
7. Describe a hyperplane, the margin and support vectors.
8. Compare linear, polynomial and RBF kernels.
9. Build leakage-free Pipelines with `StandardScaler` and compare boundaries.
10. Evaluate the test set once, after all choices are frozen.


## Required imports

Colab already includes these libraries. `%matplotlib inline` shows charts inside
the page. We set the random seeds so results repeat.


In [ ]:
import numpy as np
import matplotlib

%matplotlib inline

import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import linear_kernel as sk_linear_kernel
from sklearn.metrics.pairwise import polynomial_kernel as sk_polynomial_kernel
from sklearn.metrics.pairwise import rbf_kernel as sk_rbf_kernel

np.random.seed(42)
RANDOM_STATE = 42
print("Libraries loaded.")


## Dataset introduction

We use two datasets that are created inside the notebook, so no downloads are
needed.

| Dataset | Shape | Task |
|---|---|---|
| **Two moons** | 300 points, 2 features | Binary classification, class `0` and class `1`. |
| **Iris** | 150 flowers, 4 features | Multiclass classification, 3 species. |

The two-moons data is curved, so a straight line cannot separate the classes
well; this shows the difference between linear and nonlinear models. Iris is a
real, small flower dataset that ships with scikit-learn.

For Iris the classes are `setosa`, `versicolor` and `virginica`; each flower has
sepal length, sepal width, petal length and petal width in cm.


## What classification is

**Classification** predicts a **discrete class label** - a category, not a number
on a continuous scale. Examples: is an email spam or not; which flower species is
this; is a tumour benign or malignant.

- **Binary classification** has two labels, often written `0` and `1`. Our
  two-moons data is binary.
- **Multiclass classification** has more than two labels. Iris has three species,
  so it is multiclass.

A classifier returns a label. Some classifiers also give a **probability** for
each label, and a **threshold** turns that probability into a label.

**Small Iris example.** A flower with petal length 1.4 cm and petal width 0.2 cm
is very likely `setosa`; a flower with petal length 5.0 cm and petal width 1.7 cm
is more likely `virginica`. The model learns these label rules from labelled
examples.


In [ ]:
# Two moons: a small binary dataset with two curved classes.
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)

# Iris: a real multiclass dataset that ships with scikit-learn.
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print("moons features:", X_moons.shape, "labels:", np.unique(y_moons))
print("iris features:", X_iris.shape, "classes:", list(iris.target_names))
print("iris counts per class:", np.bincount(y_iris))


The two-moons data has 300 points with 2 features and labels `0` and `1`. Iris
has 150 flowers, 4 features and 3 balanced classes of 50 flowers each. Balanced
classes mean accuracy is a fair first metric.


### Training, validation and test sets

We split the data into three parts:

- **Training set** - the model learns from these rows.
- **Validation set** - we compare choices, such as the value of K, on these rows.
- **Test set** - we use it once, at the end, after all choices are frozen.

We must never choose K using the test set, or the test score would look too good.
We also put `StandardScaler` inside a `Pipeline` so it learns the mean and spread
from the training rows only; this avoids **data leakage**.

**Small example.** With 300 rows, we hold out 20% (60 rows) as test, then split
the remaining 240 into 180 training and 60 validation rows.


In [ ]:
# Split the row numbers first, so we can prove the parts do not overlap.
all_rows = np.arange(len(X_moons))

train_idx, test_idx = train_test_split(
    all_rows, test_size=0.20, random_state=RANDOM_STATE, stratify=y_moons
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.25, random_state=RANDOM_STATE, stratify=y_moons[train_idx]
)

X_train, y_train = X_moons[train_idx], y_moons[train_idx]
X_val, y_val = X_moons[val_idx], y_moons[val_idx]
X_test, y_test = X_moons[test_idx], y_moons[test_idx]

print("training rows:  ", X_train.shape[0])
print("validation rows:", X_val.shape[0])
print("test rows:      ", X_test.shape[0])


We have 180 training rows, 60 validation rows and 60 test rows. The three parts
do not overlap. The assertion cell near the end checks that they are disjoint and
together cover all 300 rows.


## K-Nearest Neighbours (KNN)

KNN keeps all training points. To classify a new point, it measures the
**distance** from the new point to every training point, takes the K closest ones,
and lets them vote. This is **majority voting**: the class with the most votes
wins.

A small distance means high **similarity**. The usual measure is Euclidean
distance: the straight-line distance between two points.

**Small numerical example.** Take a new point `(1, 1)` and three training points:

| Training point | Class | Euclidean distance to `(1, 1)` |
|---|---|---|
| `(1, 2)` | 0 | `sqrt(0^2 + 1^2) = 1.00` |
| `(2, 1)` | 1 | `sqrt(1^2 + 0^2) = 1.00` |
| `(3, 3)` | 1 | `sqrt(2^2 + 2^2) = 2.83` |

With `K = 3` the neighbours are all three points: two votes for class `1` and one
for class `0`, so the prediction is class `1`.

General form of Euclidean distance between points `p` and `q` with `k` features:

```text
distance = sqrt( (p1 - q1)^2 + (p2 - q2)^2 + ... + (pk - qk)^2 )
```

Because KNN uses distance, features must share one scale, so we scale them first.


In [ ]:
# Hand example: distances from a new point to three training points.
new_point = np.array([1.0, 1.0])
training_points = np.array([[1.0, 2.0], [2.0, 1.0], [3.0, 3.0]])
training_labels = np.array([0, 1, 1])

distances = np.sqrt(((training_points - new_point) ** 2).sum(axis=1))
print("distances:", np.round(distances, 2))
print("labels:   ", training_labels)

# Majority voting with K = 3.
k = 3
nearest = np.argsort(distances)[:k]
votes = np.bincount(training_labels[nearest])
print("neighbour labels:", training_labels[nearest])
print("votes per class:", votes)
print("predicted class:", int(np.argmax(votes)))


The distances are `1.00`, `1.00` and `2.83`. The three nearest neighbours have
labels `0`, `1`, `1`, so class `1` wins with two votes. A KNN classifier does the
same steps with a fast library instead of a loop.


### Choosing K on the validation set

K is a **hyperparameter**: we choose it before training. A small K follows noise;
a large K makes the boundary too smooth. We try several values, keep the one with
the best **validation accuracy**, and use the test set only once afterwards.

We have 180 training rows, so values up to about 25 still have enough neighbours
to vote.


In [ ]:
# Try several K values on the training set, score on the validation set.
k_values = list(range(1, 26, 2))
validation_scores = []

for k in k_values:
    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    knn_pipeline.fit(X_train, y_train)
    score = accuracy_score(y_val, knn_pipeline.predict(X_val))
    validation_scores.append(score)
    print(f"K = {k:2d}  validation accuracy = {score:.3f}")

best_k = k_values[int(np.argmax(validation_scores))]
print("best K on validation:", best_k)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(k_values, validation_scores, marker="o", color="tab:blue", label="Validation accuracy")
plt.axvline(best_k, color="tab:red", linestyle="--", label=f"Best K = {best_k}")
plt.title("KNN Accuracy on the Validation Set for Different K")
plt.xlabel("Number of neighbours K")
plt.ylabel("Validation accuracy")
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.show()


**Reading the chart.** Each dot is the validation accuracy for one K, and the
dashed line marks the best K. Here the best K is `1`, a very small value: small K
follows noise closely, so this win is partly luck from one validation split. The
larger, smoother K values score only a little lower. The chart still shows the
trade-off between following noise and being too smooth, and the test set will
check the frozen choice once.


## Decision trees

A decision tree asks a series of yes/no questions about the features. It has three
kinds of nodes:

- **Root node** - the first question, at the top.
- **Decision node** - an internal question that sends rows left or right.
- **Leaf node** - an end node that gives a class label.

To choose a good question, the tree measures how mixed the classes are. **Entropy**
measures that mix: `0` means one pure class, and larger values mean more mixed
classes. **Information gain** is the drop in entropy after a split; the tree picks
the split with the largest gain.

**Small numerical example before the formula.** Take 8 rows: 4 of class A and 4 of
class B. Multiply each class share by its base-2 logarithm, add the results,
then reverse the sign. Here is the arithmetic.

1. Parent, shares 4/8 and 4/8: `-(0.5 * log2(0.5) + 0.5 * log2(0.5))`. Since
   `log2(0.5) = -1`, this is `-(0.5 * -1 + 0.5 * -1) = 1.0`.
2. Split the rows into left `A=3, B=1` and right `A=1, B=3`.
3. Left child, shares 3/4 and 1/4:
   `-(0.75 * log2(0.75) + 0.25 * log2(0.25))`. Here `log2(0.75) = -0.415` and
   `log2(0.25) = -2`, so this is `-(-0.311 - 0.5) = 0.811`. The right child is the
   same by symmetry.
4. Weighted child entropy: `(4/8) * 0.811 + (4/8) * 0.811 = 0.811`.
5. Information gain: `1.0 - 0.811 = 0.189`.

General form:

```text
entropy = - sum over classes of (p_class * log2(p_class))
information gain = parent entropy - weighted average of child entropies
```

Here `p_class` is the share of rows of one class in a node, and `log2` is the
base-2 logarithm.


In [ ]:
# Entropy by hand for the small example.
def entropy(class_counts):
    counts = np.asarray(class_counts, dtype=float)
    probabilities = counts[counts > 0] / counts.sum()
    return float(-(probabilities * np.log2(probabilities)).sum())

parent_entropy = entropy([4, 4])
left_entropy = entropy([3, 1])
right_entropy = entropy([1, 3])
weighted_child = 0.5 * left_entropy + 0.5 * right_entropy
information_gain = parent_entropy - weighted_child

print("parent entropy:      ", round(parent_entropy, 3))
print("left child entropy:  ", round(left_entropy, 3))
print("right child entropy: ", round(right_entropy, 3))
print("information gain:    ", round(information_gain, 3))


The parent entropy is `1.000`, each child `0.811`, and the information gain
`0.189`, matching the hand calculation. A gain near zero means the split told us
almost nothing.


In [ ]:
# Train a decision tree on Iris using entropy, as the lesson explains.
tree_model = DecisionTreeClassifier(
    criterion="entropy", max_depth=3, random_state=RANDOM_STATE
)
tree_model.fit(X_iris, y_iris)
tree_accuracy = accuracy_score(y_iris, tree_model.predict(X_iris))
print("Iris training accuracy:", round(tree_accuracy, 3))


In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(
    tree_model,
    feature_names=iris.feature_names,
    class_names=list(iris.target_names),
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Decision Tree Trained on Iris (entropy, max_depth=3)")
plt.show()


**Reading the tree.** The top box is the root node, and it splits on petal length.
Each box shows the split rule, the entropy, the number of rows (`samples`) and the
class counts (`value`). Following the left branch leads to a pure `setosa` leaf,
which is easy to separate. The other leaves mix `versicolor` and `virginica`,
which are harder to separate. Deeper trees can fit training data better but may
overfit.


## Logistic regression

Logistic regression first computes a weighted score, then turns that score into a
**probability** between 0 and 1.

**Worked score example before the formula.** A flower has features `x1 = 2` and
`x2 = 1`. The model learned weights `w1 = 0.5`, `w2 = -1` and bias `b = 1`. The
weighted score multiplies each feature by its matching weight and adds the bias:

`z = (2 * 0.5) + (1 * -1) + 1 = 1 - 1 + 1 = 1`

The **dot product** `w . x` means "multiply matching coordinates and add them":
`(w1 * x1) + (w2 * x2)`. The **bias** `b` shifts the score up or down.

**Worked sigmoid example before the formula.** The sigmoid maps any score to a
probability:

- `z = 0` gives `1 / (1 + e^0) = 1 / (1 + 1) = 0.5`.
- `z = 2` gives `1 / (1 + e^-2) = 1 / (1 + 0.135) = 0.881`.
- `z = -2` gives `1 / (1 + e^2) = 1 / (1 + 7.389) = 0.119`.

Here `e` is Euler's number, about `2.718`, and `e^-2` means `1 / e^2`. So a large
positive `z` gives a probability near 1, and a large negative `z` gives one near 0.

General form:

```text
weighted score: z = w . x + b = w1*x1 + w2*x2 + ... + wp*xp + b
sigmoid(z) = 1 / (1 + e^-z)
probability of class 1 = sigmoid(z)
```

Symbols: `x1 ... xp` are the feature values, `w1 ... wp` their weights, `p` the
number of features, `b` the bias, `z` the weighted score, and `e` Euler's number
(about `2.718`).

**Threshold.** We turn the probability into a label: if the probability is at
least `0.5`, predict class `1`; otherwise predict class `0`. Because
`sigmoid(0) = 0.5`, the threshold `0.5` is the same as `z = 0`.

**Decision boundary.** The set of points where `z = 0` is the decision boundary,
where the model is exactly undecided. For a straight-line model this boundary is a
line.


In [ ]:
# Sigmoid function, written from the definition.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z_values = np.array([-2.0, 0.0, 2.0])
print("z:      ", z_values)
print("sigmoid:", np.round(sigmoid(z_values), 3))

z_line = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 4))
plt.plot(z_line, sigmoid(z_line), color="tab:blue", label="sigmoid(z)")
plt.axhline(0.5, color="tab:red", linestyle="--", label="threshold 0.5")
plt.axvline(0.0, color="tab:green", linestyle=":", label="z = 0")
plt.title("The Sigmoid Function")
plt.xlabel("z (linear score)")
plt.ylabel("probability")
plt.legend()
plt.grid(True)
plt.show()


**Reading the chart.** The curve starts near 0, passes through `0.5` at `z = 0`,
and rises towards 1. The red dashed line is the `0.5` threshold, so points with
`z >= 0` are predicted as class `1`. The printed values match the hand
calculation: `0.119`, `0.500`, `0.881`.


In [ ]:
# Helper that draws a model's decision boundary on the two-moons data.
def plot_boundary(ax, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    labels = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, labels, levels=[-0.5, 0.5, 1.5], alpha=0.25, cmap="coolwarm")
    ax.contour(xx, yy, labels, levels=[0.5], colors="black", linewidths=1)
    points = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    handles, _ = points.legend_elements()
    ax.legend(handles, ["Class 0", "Class 1"], loc="upper right")
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")


In [ ]:
# Train logistic regression on the two moons with a leakage-free pipeline.
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(random_state=RANDOM_STATE)),
])
logistic_model.fit(X_train, y_train)
logistic_score = accuracy_score(y_val, logistic_model.predict(X_val))
print("validation accuracy:", round(logistic_score, 3))

fig, ax = plt.subplots(figsize=(6, 5))
plot_boundary(ax, logistic_model, X_train, y_train, "Logistic Regression Boundary")
plt.tight_layout()
plt.show()


**Reading the chart.** The black line is the decision boundary where the
probability is `0.5`. It is a straight line, so logistic regression is a
**linear** classifier. The two moons are curved, so a straight line cannot
separate them well and the validation accuracy is only moderate. This is a good
reason to try a nonlinear model.


## Support Vector Machines (SVM)

An SVM separates classes with a **hyperplane**: a straight line in 2D, a plane in
3D, and a flat surface in higher dimensions. It places the hyperplane to make the
**margin** - the gap to the nearest points of each class - as wide as possible.
The points the model relies on are the **support vectors**. With a hard margin
they sit exactly on the margin edge; with a soft margin some sit inside the margin
or on the wrong side.

**Small numerical example before the formula.** Take a line through the origin
with weights `w = (1, -1)` and bias `b = 0`. The score is `z = x1 - x2`.

- Point `(2, 1)`: `z = 2 - 1 = 1 > 0`, predicted class `+1`.
- Point `(1, 2)`: `z = 1 - 2 = -1 < 0`, predicted class `-1`.
- Point `(1, 1)`: `z = 1 - 1 = 0`, exactly on the boundary.

The distance from a point to the boundary is `|z| / ||w||`, where
`||w|| = sqrt(1^2 + (-1)^2) = 1.414`. For `(2, 1)` the distance is
`1 / 1.414 = 0.707`.

General form:

```text
hyperplane: w . x + b = 0
margin width = 2 / ||w||
```

The width `2 / ||w||` holds when the two margin edges are the score levels `-1`
and `+1` (the canonical form); the distance to each edge is `1 / ||w||`. The
margin lives in the **scaled** feature space, because the `Pipeline` scales the
features before the SVM sees them; the plot below maps it back to the original
feature units.

A **hard margin** requires every point to be on the correct side. Real data has
noise, so the practical SVM uses a **soft margin**: it allows a few points inside
the margin or on the wrong side, and the penalty `C` controls how strict it is. A
large `C` punishes mistakes more and gives a narrower margin; a small `C` allows
more mistakes and gives a wider margin.

**Kernels.** A kernel measures similarity between two points. It lets the SVM draw
curved boundaries without computing the high-dimensional coordinates directly.
Three settings appear in the formulas: `degree` is the polynomial power, `gamma`
controls how far one point's influence reaches (large `gamma` = short reach and a
wiggly boundary; small `gamma` = long reach and a smooth boundary), and `coef0` is
a constant added inside the polynomial so it can mix lower-order terms.

**Small numerical examples before the formulas.** Take `x = (1, 2)` and
`z = (2, 1)`, with `gamma = 1`, `coef0 = 1` and `degree = 2`.

- **Linear kernel**: `x . z = 1*2 + 2*1 = 4`.
- **Polynomial kernel**: `(gamma * x . z + coef0)^degree = (1*4 + 1)^2 = 25`.
- **RBF kernel**: squared distance `||x - z||^2 = (1-2)^2 + (2-1)^2 = 2`, so
  `exp(-gamma * 2) = exp(-2) = 0.135`; nearby points give values near 1 and far
  points near 0.

General forms:

```text
linear kernel:      K(x, z) = x . z
polynomial kernel:  K(x, z) = (gamma * x . z + coef0)^degree
RBF kernel:         K(x, z) = exp(-gamma * ||x - z||^2)
```

The **RBF** (radial basis function) kernel creates rounded, local regions; the
**polynomial** kernel creates curved boundaries; the **linear** kernel stays a
straight line.


In [ ]:
# Verify the kernel numbers by hand and with scikit-learn.
x_point = np.array([1.0, 2.0])
z_point = np.array([2.0, 1.0])

dot_product = float(x_point @ z_point)
linear_value = dot_product
polynomial_value = (dot_product + 1.0) ** 2
rbf_value = float(np.exp(-1.0 * ((x_point - z_point) ** 2).sum()))

print("dot product:      ", dot_product)
print("linear kernel:    ", linear_value)
print("polynomial kernel:", polynomial_value)
print("RBF kernel:       ", round(rbf_value, 3))

print("sklearn linear:   ", float(sk_linear_kernel([x_point], [z_point])[0, 0]))
print("sklearn poly:     ", float(sk_polynomial_kernel(
    [x_point], [z_point], degree=2, gamma=1.0, coef0=1.0)[0, 0]))
print("sklearn RBF:      ", round(float(sk_rbf_kernel(
    [x_point], [z_point], gamma=1.0)[0, 0]), 3))


The hand values and the scikit-learn values agree: linear `4`, polynomial `25`
and RBF `0.135`. A large kernel value means two points are similar; the RBF value
shrinks quickly as points move apart.


In [ ]:
# Train a linear SVM and show its hyperplane, margin and support vectors.
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE)),
])
linear_svm.fit(X_train, y_train)

x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160))
grid = np.c_[xx.ravel(), yy.ravel()]
scores = linear_svm.decision_function(grid).reshape(xx.shape)

support = linear_svm.named_steps["svm"].support_

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, scores > 0, alpha=0.2, cmap="coolwarm")
plt.contour(
    xx, yy, scores, levels=[-1, 0, 1],
    colors=["tab:green", "black", "tab:green"], linestyles=["--", "-", "--"]
)
for class_label, color in [(0, "royalblue"), (1, "firebrick")]:
    rows = y_train == class_label
    plt.scatter(X_train[rows, 0], X_train[rows, 1], color=color, edgecolors="k", s=25, label=f"Class {class_label}")
plt.scatter(
    X_train[support, 0], X_train[support, 1], s=120,
    facecolors="none", edgecolors="tab:orange", linewidths=2, label="support vectors"
)
plt.title("Linear SVM: Hyperplane, Margin and Support Vectors")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.legend()
plt.show()

print("number of support vectors:", support.size)


**Reading the chart.** The black line is the hyperplane (`score = 0`). The two
dashed green lines are the margin edges, the score levels `-1` and `+1` mapped back
to the original feature units. The orange circles mark the support vectors: points
on the margin, points inside it, and any misclassified points. Widening the margin
means making `||w||` smaller. Because the moons are curved, a straight hyperplane
cannot separate them well even with a soft margin, so several support vectors
violate the margin.


In [ ]:
# Train one SVM per kernel and score each on the validation set.
svm_models = {
    "SVM linear": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE)),
    ]),
    "SVM polynomial": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="poly", degree=3, coef0=1.0, C=1.0, random_state=RANDOM_STATE)),
    ]),
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", gamma="scale", C=1.0, random_state=RANDOM_STATE)),
    ]),
}

for name, model in svm_models.items():
    model.fit(X_train, y_train)
    score = accuracy_score(y_val, model.predict(X_val))
    print(f"{name:16s} validation accuracy = {score:.3f}")

# Check that every kernel was fitted.
assert all(model.named_steps["svm"].support_.size > 0 for model in svm_models.values())
print("all kernels trained")


All three kernels trained. The linear kernel is limited to a straight boundary, so
it scores lowest. The polynomial and RBF kernels can bend, and here the polynomial
kernel has the best validation score. A small difference on one validation split
is not a promise for the test set, which we use once later.


### Comparing classifier boundaries

Now we place the main classifiers side by side on the same training data. Each
panel shows the region predicted for class `0` and class `1`, and the black line
is the boundary where the model changes its mind. Comparing panels shows which
models can bend and which cannot.


In [ ]:
# One KNN model using the best K from validation, plus a tree on the moons.
best_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])
best_knn.fit(X_train, y_train)

tree_moons = Pipeline([
    ("scaler", StandardScaler()),
    ("tree", DecisionTreeClassifier(
        criterion="entropy", max_depth=3, random_state=RANDOM_STATE
    )),
])
tree_moons.fit(X_train, y_train)

comparison_models = {
    f"KNN (K={best_k})": best_knn,
    "Decision tree": tree_moons,
    "Logistic regression": logistic_model,
    "SVM linear": svm_models["SVM linear"],
    "SVM polynomial": svm_models["SVM polynomial"],
    "SVM RBF": svm_models["SVM RBF"],
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, model) in zip(axes.ravel(), comparison_models.items()):
    plot_boundary(ax, model, X_train, y_train, name)
plt.suptitle("Classifier Decision Boundaries on the Two Moons (training data)")
plt.tight_layout()
plt.show()


**Reading the grid.** KNN draws a flexible, jagged boundary. The decision tree
draws straight, box-like steps because it splits one feature at a time. Logistic
regression and the linear SVM both draw one straight line, so they underfit the
curved moons. The polynomial SVM bends, and the RBF SVM makes smooth, rounded
regions that follow the moons best here. No single model is always best; the
shape of the data decides.


### Freezing the choices and using the test set once

We choose the final model using the **validation** set only. The next cell scores
every candidate on the validation rows and picks the best one; the test rows are
not used yet. Only after that choice is frozen do we evaluate the single selected
model on the test set, once.


In [ ]:
# Choose the final model using the validation set only.
validation_results = {}
for name, model in comparison_models.items():
    validation_results[name] = accuracy_score(y_val, model.predict(X_val))

for name, score in validation_results.items():
    print(f"{name:20s} validation accuracy = {score:.3f}")

selected_model_name = max(validation_results, key=validation_results.get)
selected_model = comparison_models[selected_model_name]
print("selected model (by validation):", selected_model_name)


In [ ]:
# Evaluate ONLY the selected model on the test set, once.
test_accuracy = accuracy_score(y_test, selected_model.predict(X_test))
print("selected model:", selected_model_name)
print("test accuracy: ", round(test_accuracy, 3))


The selected model was chosen on the validation set, so its test score is an
honest estimate. We do not rank the other models on test and we do not tune
anything after seeing this number. If the test score were much lower than
validation, that would warn about overfitting to the validation set.


In [ ]:
# Concise checks: entropy, sigmoid, splits and trained kernels.
assert np.isclose(parent_entropy, 1.0)
assert np.isclose(left_entropy, 0.811278, atol=1e-5)
assert np.isclose(information_gain, 1.0 - 0.811278, atol=1e-5)
assert np.isclose(sigmoid(0.0), 0.5)
assert np.isclose(sigmoid(2.0), 0.880797, atol=1e-5)

# The three splits must be disjoint and cover every row.
assert set(train_idx).isdisjoint(set(val_idx))
assert set(train_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))
assert len(set(train_idx) | set(val_idx) | set(test_idx)) == len(X_moons)

# Every kernel model must be trained.
assert all(model.named_steps["svm"].support_.size > 0 for model in svm_models.values())

# The final model was chosen on validation and scored on test exactly once.
assert selected_model_name in comparison_models
assert 0.0 <= test_accuracy <= 1.0

# Both decision trees must use entropy, as the lesson explains.
assert tree_model.criterion == "entropy"
assert tree_moons.named_steps["tree"].criterion == "entropy"

print("all checks passed")


## Common mistakes

- **Choosing K or the final model on the test set.** Use the validation set for
  every choice and the test set once, for the chosen model only.
- **Scaling before splitting.** Fit `StandardScaler` inside a `Pipeline` on
  training data only; otherwise test information leaks in.
- **Confusing entropy with accuracy.** Entropy measures how mixed a node is, not
  how correct a model is.
- **Forgetting that KNN and SVM need scaled features.** Distance-based models
  change a lot when units differ.
- **Using a linear model on curved data.** Logistic regression and a linear SVM
  draw straight boundaries; curved data needs a kernel or a tree.
- **Reading a wide margin as always better.** A very wide soft margin can ignore
  real patterns; the penalty `C` balances margin width and mistakes.
- **Thinking a leaf must be pure.** A tree stopped early (`max_depth`) can leave
  mixed leaves; the leaf then predicts the majority class.


## Summary

- **Classification** predicts discrete labels; **binary** has two labels and
  **multiclass** more than two (Iris has three).
- **KNN** uses **distance** and **majority voting**; we chose K on the validation
  set and read the accuracy-versus-K chart.
- **Decision trees** split with **root**, **decision** and **leaf** nodes, using
  **entropy** and **information gain**; we saw a hand entropy example and the Iris
  tree picture.
- **Logistic regression** turns a linear score into a **probability** with the
  **sigmoid**, and the `0.5` **threshold** gives the **decision boundary**.
- **SVMs** use a **hyperplane** with a **margin** and **support vectors**;
  **linear**, **polynomial** and **RBF kernels** allow different boundary shapes,
  and a **soft margin** tolerates noise.
- We used clear **train, validation and test** splits, kept `StandardScaler`
  inside a **Pipeline**, and used the test set once after freezing our choices.
